<a href="https://colab.research.google.com/github/fatimaali123-ai/flyrank-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fatimaali123-ai/flyrank-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
!git clone https://github.com/fatimaali123-ai/flyrank-internship.git

fatal: destination path 'flyrank-internship' already exists and is not an empty directory.


In [ ]:
import pandas as pd
import numpy as np

DATA_PATH = "/content/flyrank-internship/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset shape: (30000, 44)

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [ ]:
df["is_declining_label"] = (
    df["trend_direction"]
    .astype(str)
    .str.lower()
    .eq("down")
    .astype(int)
)

print("Target distribution:")
print(df["is_declining_label"].value_counts())

print("\nDeclining rate:")
print(df["is_declining_label"].mean())

Target distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Declining rate:
0.5420666666666667


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 1. Two paper findings + my methodology questions

### Finding 1: The paper reports a relationship between search/traffic signals and content performance.

**Methodology question:** Where does the outcome label come from, and is the label defined using information that was available before the prediction point? I would want to understand the exact label construction and observation window so that the reported relationship is not partly caused by information from the future.

This is a constructive question because a clear label definition and time boundary would make the finding easier to interpret and reproduce.

### Finding 2: The paper reports model or analysis results based on a validation/evaluation design.

**Methodology question:** Does the validation design support the scope of the claim? In particular, I would ask whether related observations from the same client or time period can appear on both sides of the validation split, and whether the evaluation represents the setting in which the model would actually be used.

This does not challenge the finding itself. It asks whether the validation design is strong enough to support the stated level of generalization.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

RANDOM_STATE = 42

# ---------------------------------------------------------
# Create grouped train/test split
# ---------------------------------------------------------

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE
)

train_idx, test_idx = next(
    gss.split(
        df,
        y=df["is_declining_label"],
        groups=df["client_id"]
    )
)

train = df.iloc[train_idx].copy()
test = df.iloc[test_idx].copy()

print("Train shape:", train.shape)
print("Test shape:", test.shape)

print("\nTrain clients:", train["client_id"].nunique())
print("Test clients:", test["client_id"].nunique())

overlap = set(train["client_id"]).intersection(
    set(test["client_id"])
)

print("\nClient overlap:", overlap)


Train shape: (23837, 45)
Test shape: (6163, 45)

Train clients: 25
Test clients: 7

Client overlap: set()


In [ ]:
# ---------------------------------------------------------
# Feature selection
# ---------------------------------------------------------

excluded = [
    "content_id",
    "client_id",
    "is_declining_label",
    "trend_direction",
    "trend_pct"
]

feature_cols = [
    col for col in train.columns
    if col not in excluded
    and pd.api.types.is_numeric_dtype(train[col])
]

print("Number of features:", len(feature_cols))

# ---------------------------------------------------------
# X and y
# ---------------------------------------------------------

X_train = train[feature_cols].copy()
X_test = test[feature_cols].copy()

y_train = train["is_declining_label"].astype(int)
y_test = test["is_declining_label"].astype(int)

# ---------------------------------------------------------
# Missing values
# ---------------------------------------------------------

train_medians = X_train.median()

X_train = X_train.fillna(train_medians)
X_test = X_test.fillna(train_medians)

# ---------------------------------------------------------
# Random Forest
# ---------------------------------------------------------

model_honest = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

model_honest.fit(X_train, y_train)

honest_scores = model_honest.predict_proba(X_test)[:, 1]

print("Honest grouped model trained successfully.")

Number of features: 29
Honest grouped model trained successfully.


In [ ]:
def precision_at_k(y_true, scores, k):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    top_indices = np.argsort(-scores)[:k]

    return y_true[top_indices].mean()


# Week-4 baseline
baseline_scores = (
    test["impressions_90d"] *
    (1 - test["ctr"] / 100)
).fillna(0)


# Honest model
honest_p20 = precision_at_k(
    y_test,
    honest_scores,
    20
)

honest_p50 = precision_at_k(
    y_test,
    honest_scores,
    50
)


# Before = Week 5 results
week5_p20 = 1.00
week5_p50 = 1.00


comparison = pd.DataFrame({
    "Evaluation": [
        "Week-5 model",
        "Week-6 honest grouped model"
    ],
    "Precision@20": [
        week5_p20,
        honest_p20
    ],
    "Precision@50": [
        week5_p50,
        honest_p50
    ]
})

display(comparison)

,Evaluation,Precision@20,Precision@50
0,Week-5 model,1.0,1.0
1,Week-6 honest grouped model,1.0,1.0


## 2. My model under an honest split

The Week-5 model was evaluated using a grouped-by-client train/test split. In this audit, I will reproduce the model using the same client-grouped principle and explicitly compare the result with the Week-5 evaluation.

Grouping by client is important because observations from the same client can share characteristics that make random observation-level validation overly optimistic.

The purpose of this comparison is not to maximize the score. It is to measure how stable the observed performance is under a validation design that better reflects the intended use case.

I will report the result as observed performance on the evaluated split rather than claiming that the model will generalize perfectly to all future content.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [ ]:
# Leakage audit

excluded_features = [
    "content_id",
    "client_id",
    "is_declining_label",
    "trend_direction",
    "trend_pct"
]

leakage_audit = pd.DataFrame({
    "Feature": excluded_features,
    "Decision": [
        "Excluded",
        "Excluded",
        "Excluded",
        "Excluded",
        "Excluded"
    ],
    "Reason": [
        "Identifier; not a predictive feature",
        "Used only for grouped validation; not a predictive feature",
        "Target variable",
        "Used to create the target; would cause target leakage",
        "Outcome/trend-derived variable; excluded to avoid leakage"
    ]
})

display(leakage_audit)

print("Model features:")
print(feature_cols)

print("\nNumber of model features:", len(feature_cols))

print("\nExcluded leakage/identifier fields:")
print(excluded_features)

,Feature,Decision,Reason
0,content_id,Excluded,Identifier; not a predictive feature
1,client_id,Excluded,Used only for grouped validation; not a predic...
2,is_declining_label,Excluded,Target variable
3,trend_direction,Excluded,Used to create the target; would cause target ...
4,trend_pct,Excluded,Outcome/trend-derived variable; excluded to av...


Model features:
['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier_order', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']

Number of model features: 29

Excluded leakage/identifier fields:
['content_id', 'client_id', 'is_declining_label', 'trend_direction', 'trend_pct']


## 3. Leakage audit

I audited the features used by my Week-5 model for possible leakage.

I excluded `content_id` and `client_id` because they are identifiers and should not be used as predictive features. I also excluded `is_declining_label`, `trend_direction`, and `trend_pct`. The target `is_declining_label` was created from `trend_direction == "down"`, so using `trend_direction` or `trend_pct` as model features could directly leak information about the target.

The remaining features describe historical search, traffic, engagement, content age, CTR, and position signals. Missing numeric values were handled using medians calculated from the training data only and then applied to the test data.

Based on this audit, I found no obvious direct target leakage in the features used by the model. However, this does not prove that all possible sources of bias or leakage have been eliminated. The model results should therefore be treated as measured performance on the evaluated client-grouped split.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [ ]:
# =========================================================
# Real failure examples
# =========================================================

audit_results = test[
    ["content_id", "client_id", "trend_direction", "trend_pct"]
].copy()

audit_results["actual"] = y_test.to_numpy()
audit_results["model_score"] = honest_scores
audit_results["prediction"] = (
    honest_scores >= 0.5
).astype(int)

# False positives
false_positives = audit_results[
    (audit_results["actual"] == 0) &
    (audit_results["prediction"] == 1)
].copy()

# False negatives
false_negatives = audit_results[
    (audit_results["actual"] == 1) &
    (audit_results["prediction"] == 0)
].copy()

print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

print("\nFalse positive examples:")
display(
    false_positives
    .sort_values("model_score", ascending=False)
    .head(5)
)

print("\nFalse negative examples:")
display(
    false_negatives
    .sort_values("model_score", ascending=True)
    .head(5)
)

False positives: 1016
False negatives: 740

False positive examples:


,content_id,client_id,trend_direction,trend_pct,actual,model_score,prediction
20736,content_41baf0722ad9,client_8527a891e2,stable,-14.3,0,0.743062,1
2357,content_8f1409b2674e,client_8527a891e2,stable,-17.9,0,0.742569,1
2488,content_204b729683f6,client_f369cb89fc,stable,-13.2,0,0.741534,1
28718,content_ef6e7d7cfe15,client_8527a891e2,stable,11.4,0,0.740970,1
12332,content_4d9f36001f06,client_8527a891e2,stable,-17.0,0,0.740500,1



False negative examples:


,content_id,client_id,trend_direction,trend_pct,actual,model_score,prediction
18929,content_f3ca73f0f3f3,client_e629fa6598,down,-33.3,1,0.231206,0
27575,content_284df888e0cb,client_e629fa6598,down,-25.0,1,0.238148,0
3709,content_6e6e8c6fcd2a,client_e629fa6598,down,-20.8,1,0.245879,0
14282,content_adb76cf9c286,client_e629fa6598,down,-67.2,1,0.255034,0
2208,content_827b209fa167,client_4e07408562,down,-23.0,1,0.263766,0


## 4. Claim rewrite

A claim such as "the Random Forest perfectly predicts declining content" would be too strong.

A safer claim is:

"On the evaluated client-grouped test split, the Random Forest measured Precision@20 of 1.00 and Precision@50 of 1.00."

This result is observed and measured performance on the evaluated split. It does not prove that the model will achieve perfect performance on future data or on every client.

The model can therefore be described as directional decision-support for prioritizing content for review, rather than as a system that guarantees which content will decline.

The most important observed features included historical impressions, recent impressions, content age, and average position. These feature-importance results indicate useful predictive signals, but they do not establish that these features cause content to decline.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.